# Global Air Quality Analysis & Prediction

**Dataset:** Global Air Pollution Dataset (Kaggle)  
**Goal:** Explore global air quality patterns and build ML models to predict AQI values based on pollutant concentrations.

### Techniques Used
- Exploratory Data Analysis (EDA)
- Feature Engineering
- Regression Models: Linear Regression, Random Forest, XGBoost
- Model Evaluation: MAE, RMSE, R²

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')

print('Libraries loaded successfully')

## 2. Load & Inspect Data

In [ ]:
df = pd.read_csv('data/global_air_pollution.csv')

print('Shape:', df.shape)
print('\nColumn names:', df.columns.tolist())
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Data Cleaning

In [ ]:
# Check missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0])

# Drop duplicates
before = len(df)
df = df.drop_duplicates()
print(f'\nDropped {before - len(df)} duplicate rows')

# Drop rows with missing AQI (target variable)
df = df.dropna(subset=['AQI Value'])
print(f'Final shape: {df.shape}')

## 4. Exploratory Data Analysis

In [ ]:
# Distribution of AQI values
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['AQI Value'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of AQI Values')
axes[0].set_xlabel('AQI Value')
axes[0].set_ylabel('Count')

aqi_counts = df['AQI Category'].value_counts()
axes[1].bar(aqi_counts.index, aqi_counts.values, color='coral', edgecolor='white')
axes[1].set_title('AQI Category Distribution')
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# Top 15 most polluted countries (median AQI)
top_countries = (
    df.groupby('Country')['AQI Value']
    .median()
    .sort_values(ascending=False)
    .head(15)
)

plt.figure(figsize=(12, 5))
top_countries.plot(kind='bar', color='tomato', edgecolor='white')
plt.title('Top 15 Most Polluted Countries (Median AQI)')
plt.xlabel('Country')
plt.ylabel('Median AQI')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap — which pollutants drive AQI?
numeric_cols = ['AQI Value', 'CO AQI Value', 'Ozone AQI Value', 'NO2 AQI Value', 'PM2.5 AQI Value']
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    square=True,
    linewidths=0.5
)
plt.title('Pollutant Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots: each pollutant vs AQI
pollutants = ['CO AQI Value', 'Ozone AQI Value', 'NO2 AQI Value', 'PM2.5 AQI Value']

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, col in enumerate(pollutants):
    axes[i].scatter(df[col], df['AQI Value'], alpha=0.3, s=10, color='steelblue')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('AQI Value')
    axes[i].set_title(f'{col} vs AQI')

plt.suptitle('Pollutant vs AQI Scatter Plots', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 5. Feature Engineering

In [ ]:
# Encode AQI Category as a numeric feature
category_order = {'Good': 0, 'Moderate': 1, 'Unhealthy for Sensitive Groups': 2,
                  'Unhealthy': 3, 'Very Unhealthy': 4, 'Hazardous': 5}
df['AQI Category Encoded'] = df['AQI Category'].map(category_order)

# Dominant pollutant: which pollutant has the highest AQI contribution per row
df['Dominant Pollutant AQI'] = df[pollutants].max(axis=1)

# Pollution load: sum of all pollutant AQI values
df['Total Pollutant Load'] = df[pollutants].sum(axis=1)

print('New features added:')
print(df[['AQI Category Encoded', 'Dominant Pollutant AQI', 'Total Pollutant Load']].head())

## 6. Model Building

We will predict **AQI Value** using pollutant sub-index values and engineered features.  
Three models are compared: Linear Regression (baseline), Random Forest, and XGBoost.

In [ ]:
features = pollutants + ['AQI Category Encoded', 'Dominant Pollutant AQI', 'Total Pollutant Load']
target = 'AQI Value'

X = df[features].fillna(0)
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
def evaluate_model(name, y_true, y_pred):
    """Print MAE, RMSE, and R² for a model."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f'{name:<25} MAE: {mae:.2f}   RMSE: {rmse:.2f}   R²: {r2:.4f}')
    return {'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2}

results = []

In [ ]:
# --- Baseline: Linear Regression ---
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_preds = lr.predict(X_test)
results.append(evaluate_model('Linear Regression', y_test, lr_preds))

In [ ]:
# --- Random Forest ---
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_preds = rf.predict(X_test)
results.append(evaluate_model('Random Forest', y_test, rf_preds))

In [ ]:
# --- XGBoost ---
try:
    from xgboost import XGBRegressor
    xgb = XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6,
                       random_state=42, verbosity=0)
    xgb.fit(X_train, y_train)
    xgb_preds = xgb.predict(X_test)
    results.append(evaluate_model('XGBoost', y_test, xgb_preds))
except ImportError:
    print('XGBoost not installed. Run: pip install xgboost')
    xgb_preds = None

## 7. Model Comparison

In [ ]:
results_df = pd.DataFrame(results)
results_df = results_df.set_index('Model')
print('\n=== Model Comparison ===')
print(results_df.round(4))

In [ ]:
# Actual vs Predicted — best model
best_preds = xgb_preds if xgb_preds is not None else rf_preds
best_name = 'XGBoost' if xgb_preds is not None else 'Random Forest'

plt.figure(figsize=(8, 6))
plt.scatter(y_test, best_preds, alpha=0.3, s=10, color='steelblue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
         color='red', linestyle='--', linewidth=1.5, label='Perfect Prediction')
plt.title(f'{best_name}: Actual vs Predicted AQI')
plt.xlabel('Actual AQI')
plt.ylabel('Predicted AQI')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance — Random Forest
importances = pd.Series(rf.feature_importances_, index=features).sort_values(ascending=True)

plt.figure(figsize=(8, 5))
importances.plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('Feature Importance (Random Forest)')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

## 8. Findings & Conclusion

**Key findings:**
- PM2.5 is the dominant driver of AQI values globally, showing the strongest correlation.
- XGBoost outperforms both Linear Regression and Random Forest in RMSE and R².
- The `Total Pollutant Load` engineered feature ranked highly in feature importance, confirming that combined pollutant exposure better explains overall air quality than any single pollutant.

**Next steps:**
- Incorporate geographic/regional encoding for country-level effects.
- Explore time-series forecasting if temporal data becomes available.